# Inference and Information Extraction

This notebook implements the **inference stage** of the Emergency Care Disruption Dataset (ECDD) pipeline.

In the previous stage of the pipeline, a web retrieval process was used to collect documents related to French hospital emergency departments using the Tavily Search API. Each hospital in the FINESS registry was used to generate targeted queries, producing a set of candidate sources (news articles, institutional communications, and press releases).

The objective of this notebook is to **analyze the retrieved documents and extract structured information about emergency department disruptions**.

Using a local large language model (**Mistral-7B-Instruct**, deployed with **vLLM**), the pipeline processes the textual content of each source and identifies relevant events such as:

- temporary emergency department closures  
- regulated access to emergency services  
- strikes affecting emergency care  
- SMUR service disruptions  

The extracted information is then converted into structured records that can be stored in a long-format dataset where **each row corresponds to a single event supported by a verifiable source URL**.

Within the overall pipeline, this notebook corresponds to the **inference** stage of the pipeline, transforming unstructured web content into machine-readable data for subsequent structuring and analysis.

In [1]:
import json
import time
import os
import re
import pandas as pd
from datetime import datetime
from vllm import LLM, SamplingParams

ideas:
1) https://www.youtube.com/watch?app=desktop&v=9j-480mlXEk&start=0

## Local LLM Setup: Why Mistral-7B and Why vLLM?

This notebook uses a **local large language model**, `Mistral-7B-Instruct-v0.3`, served through **vLLM**, to perform structured information extraction from the retrieved web sources. 

### Why use a local model?

A local model was preferred over an external API for three main reasons:

1. **Reproducibility**  
   Running the model locally makes the extraction pipeline easier to reproduce, since the same model version and inference settings can be reused across runs.

2. **Auditability and control**  
   The inference process remains fully controlled within the project environment. This is particularly important when processing hospital-level evidence and preserving a transparent extraction pipeline.

3. **Scalability for batch processing**  
   The task requires repeated inference over many retrieved documents. A local setup allows large batches of extractions without depending on API quotas or external service availability.

### Why `Mistral-7B-Instruct-v0.3`?

[`Mistral-7B-Instruct-v0.3`](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) was selected as a practical compromise between **instruction-following ability**, **computational efficiency**, and **local deployability**.


This model is suitable for the current task because:

- it can follow structured prompts reliably,
- it is lightweight enough to run on local GPU infrastructure,
- it performs well for extraction-style tasks where the goal is to return concise, schema-constrained outputs rather than long-form generation.

In this notebook, the model is not used for open-ended text generation, but as a **controlled extractor** of event attributes from retrieved sources.

### Why use vLLM?

The model is served through [`vLLM`](https://github.com/vllm-project/vllm), an inference engine designed for efficient deployment of large language models.

vLLM is used here because it provides:

- **fast inference** for repeated prompt execution,
- **efficient GPU memory management**,
- a simple interface for running the model locally in batch settings.

This makes it well suited for a pipeline in which many documents must be processed sequentially or in batches.

### Inference settings

The model is loaded from disk and configured with the following parameters:

- `dtype='half'`  
  Uses half-precision floating point format to reduce GPU memory usage.

- `gpu_memory_utilization=0.5`  
  Limits the fraction of GPU memory allocated to the model, helping avoid memory saturation.

- `max_model_len=2000`  
  Restricts the maximum input context length, which is sufficient for the extraction prompts used in this notebook while remaining computationally manageable.

- `temperature=0`  
  Forces deterministic generation, which is appropriate for structured extraction tasks where consistency is preferred over creativity.

- `max_tokens=256`  
  Caps the output length, since the expected output is a short structured response rather than free-form text.



In [ ]:
# Force vLLM to use the Data partition for everything
os.environ['VLLM_CONFIG_ROOT'] = '/Data/anahi_reyes/vllm_cache/config'
os.environ['VLLM_CACHE_ROOT'] = '/Data/anahi_reyes/vllm_cache'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['HF_HOME'] = '/Data/anahi_reyes/huggingface_cache'
# Forzamos el uso del motor estable (V0)
#os.environ["VLLM_USE_V1"] = "false"
# También por seguridad, evitamos que intente usar HTTP moderno si hay conflictos de red local
#os.environ["VLLM_CONFIGURE_LOGGING"] = "1"

# Load the in-disk LLM to the environment
model_path = '/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3'
llm = LLM(model=model_path, dtype='half', gpu_memory_utilization=0.8, max_model_len=4096)
sampling_params = SamplingParams(temperature=0, max_tokens=512,  stop=["</json>", "[/INST]"])


INFO 03-14 09:49:37 [utils.py:261] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': '/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3'}
INFO 03-14 09:49:37 [model.py:541] Resolved architecture: MistralForCausalLM
WARNING 03-14 09:49:37 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 03-14 09:49:37 [model.py:1561] Using max model len 4096
INFO 03-14 09:49:37 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-14 09:49:37 [vllm.py:624] Asynchronous scheduling is enabled.


Multiple tokenizer files found in directory: /Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3. Using tokenizer.model.v3.


(EngineCore_DP0 pid=812977) INFO 03-14 09:49:38 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endp

(EngineCore_DP0 pid=812977) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore_DP0 pid=812977) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=812977) INFO 03-14 09:49:46 [default_loader.py:291] Loading weights took 3.12 seconds
(EngineCore_DP0 pid=812977) INFO 03-14 09:49:47 [gpu_model_runner.py:4118] Model loading took 13.51 GiB memory and 4.105459 seconds
(EngineCore_DP0 pid=812977) INFO 03-14 09:49:58 [backends.py:805] Using cache directory: /Data/anahi_reyes/vllm_cache/torch_compile_cache/6563c897c5/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=812977) INFO 03-14 09:49:58 [backends.py:865] Dynamo bytecode transform time: 10.71 s
(EngineCore_DP0 pid=812977) INFO 03-14 09:50:02 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.514 s
(EngineCore_DP0 pid=812977) INFO 03-14 09:50:02 [monitor.py:34] torch.compile takes 12.22 s in total
(EngineCore_DP0 pid=812977) INFO 03-14 09:50:03 [gpu_worker.py:356] Available KV cache memory: 4.42 GiB
(EngineCore_DP0 pid=812977) INFO 03-14 09:50:03 [kv_cache_utils.py:1307] GPU KV cache size: 36,240 t

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 15.15it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 19.82it/s]


(EngineCore_DP0 pid=812977) INFO 03-14 09:50:09 [gpu_model_runner.py:5051] Graph capturing finished in 6 secs, took 0.51 GiB
(EngineCore_DP0 pid=812977) INFO 03-14 09:50:09 [core.py:272] init engine (profile, create kv cache, warmup model) took 22.02 seconds


(EngineCore_DP0 pid=812977) Multiple tokenizer files found in directory: /Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3. Using tokenizer.model.v3.


INFO 03-14 09:50:09 [llm.py:343] Supported tasks: ['generate']


In [40]:
df = df = pd.read_json("/Data/anahi_reyes/EDCD_data/raw_edcd_database_atomic.jsonl", lines=True)
df.head()

,finess,hospital_name,nom_etab_long,keywords_nom_etab,source_url,title,content,score,retrieved_at
0,910300219,HOPITAL PRIVE JACQUES CARTIER,HOPITAL PRIVE JACQUES CARTIER,JACQUES CARTIER,https://hopital-prive-jacques-cartier-massy.ramsaysante.fr/vous-%C3%AAtes-patient-pourquoi-choisir-notre-%C3%A9tablissement/urgences-5,Prise en charge en urgence | Hôpital privé Jacques Cartier,"Le délai d'attente aux urgences constitue un indice majeur de l'organisation du service et sur la qualité de la prise en charge des patients. C'est la raison pour laquelle le Groupe a mis en place depuis 2013 un projet de transparence et d'affichage des délais d'attente aux urgences dans plusieurs de ses établissements. Ce dispositif innovant permet de mesurer et d'afficher le délai d'attente des patients sur le site Internet des cliniques et hôpitaux privés concernés. Plébiscitée par les patients, la démarche a vocation à être étendue à l'ensemble des services d'urgences Ramsay Générale de Santé. Elle s'appuie sur une organisation repensée des services et des flux de patients. [...] En novembre 2013, Lemonde.fr publie le reportage de Jean-Baptiste Jacquin consacré aux urgences de nuit de l'hôpital privé de Seine-Saint-Denis, au Blanc-Mesnil. Durant 4 jours, le journaliste, accompagné d'un photographe, a enquêté au sein de l'établissement. Dans son article, le reporter souligne la qualité du service rendu à la population d'un département socialement défavorisé : 18 % des patients accueillis par l'établissement sont bénéficiaires de la couverture médicale universelle (CMU) ou de l'aide médicale d'État (AME). Afin de mieux informer et comprendre ses patients non francophones, l'hôpital a constitué un pool d'interprètes au sein de ses équipes.\n\nAucun dépassement d'honoraires n'est pratiqué aux urgences.\n\nRecrutement\nTrouver un médecin\nNous contacter\nPaiement en ligne [...] ## Nos services d’urgence\n\nDepuis 2007, les urgences constituent une activité soumise à autorisation dans laquelle le secteur public et le secteur privé sont assujettis au même cahier des charges réglementaire. Précurseur dans ce domaine parmi les opérateurs privés, Ramsay Générale de Santé contribue significativement, aux côtés des hôpitaux publics, à couvrir les besoins médicaux d'urgence de la population. Ses 23 services de soins d'urgence, répartis sur tout le territoire, accueillent près de 486 000 patients par an 24h/24 et 7 jours/7. Pris en charge par une équipe soignante spécialisée, chaque patient bénéficie de tous les moyens d'investigation nécessaires à sa prise en charge.\n\n## Reportage aux urgences de l'Hôpital privé de la Seine-Saint-Denis",0.999846,2026-03-14 11:23:51
1,910300219,HOPITAL PRIVE JACQUES CARTIER,HOPITAL PRIVE JACQUES CARTIER,JACQUES CARTIER,https://igas.gouv.fr/sites/igas/files/2024-03/Evaluation%20des%20mesures%20d%C3%A9rogatoires%20portant%20sur%20les%20soins%20urgents%20et%20non%20programm%C3%A9s%20pour%20l%E2%80%99%C3%A9t%C3%A9%202022.pdf,[PDF] Evaluation des mesures dérogatoires portant sur les soins urgents ...,"que « Les fermetures totales enregistrées ont touché la plupart du temps des établissements sur des dates isolées, où l’établissement n’a pas pu trouver les ressources pour maintenir un accueil. » RAPPORT IGAS N°2022-064R - 18 - Carte 1 : Part des structures d’urgence complètement ou partiellement fermées le 1er septembre 2022 sur celles installées à date (%) Source : DGOS Carte 2 : SU en fermeture partielle (nuits) Source : Enquête ARS, exploitation IGAS 2.2 L’activité des SU régulés : Données issues notamment de l’enquête ORU Une enquête a été diligentée auprès des Observatoires régionaux des urgences (cf. annexe), visant à apprécier l’impact des dispositifs de régulation sur l’activité des services, au regard de cinq critères : activité du SU régulé, report éventuel d’activité sur les [...] mission 2.1.2 Données sur les fermetures partielles ou totales de SU La mise en place de régulation à l’accès aux urgences en permettant de dimi

In [41]:
def build_extraction_prompt(row):
    h_name = row['hospital_name']
    h_keywords = row['keywords_nom_etab']
    return f"""<s>[INST]
You are a structured data extraction assistant specializing in French healthcare disruptions.

TARGET HOSPITAL:
- Official name: "{h_name}"
- Also known as: "{h_keywords}"
- Match even with abbreviations or partial names.
- Do NOT match other hospitals sharing only a generic word like "CHU" or "Centre".

TASK:
Determine whether THIS TARGET HOSPITAL's emergency department (Urgences) is experiencing
a confirmed disruption, a potential disruption, or neither.

Set relevant = true if the article explicitly states that the urgences are:
- Closed (fully or partially, day or night)
- Operating with reduced staffing or reorganized access
- Regulated (patients must call before coming)
- Redirecting ambulances
- The article describes a past disruption that has now ended — the event still occurred and should be captured.

Set relevant = "potential" if no disruption has been implemented yet, but the article
explicitly describes a direct threat to the urgences:
- Documented staff shortage or upcoming reorganization
- ARS intervention or administrative crisis specifically affecting the urgences
- Political motions or demands specifically about the urgences

Set relevant = false in all other cases, including:
- General hospital tensions without specific urgences impact
- Disruptions in other departments (surgery, maternity, psychiatry, etc.)
- The hospital appears only as a backup/receiving hospital
- Proposals or debates with nothing implemented yet

When in doubt, prefer false over true or potential.

ROLE IDENTIFICATION:
STEP 1: Copy the exact name of the hospital whose urgences are closing, regulated, or at risk.
STEP 2: Does that name contain "{h_name}" or "{h_keywords}"?
        - Yes → proceed to assess relevant = true or "potential"
        - No → relevant = false, regardless of whether the target hospital is mentioned elsewhere.
STEP 3: If the TARGET HOSPITAL is mentioned only as receiving patients, it is OPEN → relevant = false.

TEMPORAL REASONING:
- The publication date may appear in the URL (e.g. /2024/03/15/ or /2024-03-15/).
- If only month and year are known, use the first of the month: "2022-07-01".
- If found in the URL, use it as the publication date.
- If also present in the article text, prefer the article text date.
- WARNING: ignore dates found in unrelated links, footers, or "related articles" sections.
- Use it to resolve relative expressions: "today", "this weekend", "from Monday".
- If no publication date is found, leave relative expressions as "Unknown" — do not guess.
- end_date = "Ongoing" only if the article explicitly says no end date is set.
- end_date = "Unknown" if a closure is mentioned but no end date is given.
- end_date = "YYYY-MM-DD" if a specific date is stated or clearly implied.

OUTPUT:
First write 1-2 sentences explaining your decision.
Then output the result in a <json> block.

Confirmed disruption case:
<json>
{{
  "relevant": true,
  "status": "choose one: Fermeture complète / Fermeture temporaire / Régulation des entrées / Détournement ambulances / Unknown",
  "start_date": "YYYY-MM-DD | Unknown",
  "end_date": "YYYY-MM-DD | Ongoing | Unknown",
  "reason": "Free text — describe the cause of the disruption as mentioned. Unknown if not mentioned.",
  "publication_date": "YYYY-MM-DD | Unknown",
  "confidence": "High | Medium | Low"
}}
</json>

Potential disruption case:
<json>
{{
  "relevant": "potential",
  "risk_description": "Free text — describe what situation puts the urgences at risk.",
  "publication_date": "YYYY-MM-DD | Unknown",
  "confidence": "High | Medium | Low"
}}
</json>

Non-relevant case:
<json>{{"relevant": false}}</json>

TEXT:
URL: {row['source_url']}
Title: {row['title']}
Content: {row['content']}
[/INST]"""

def robust_json_parse(raw_output):
    # Try <json> tag first (structured output)
    match = re.search(r'<json>(.*?)</json>', raw_output, re.DOTALL)
    if not match:
        # Fallback to bare JSON object
        match = re.search(r'(\{.*\})', raw_output, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1).strip())
        except json.JSONDecodeError as e:
            return {"relevant": False, "parsing_error": True, "error_msg": str(e), "raw": raw_output[:200]}
    return {"relevant": False, "no_json_found": True, "raw": raw_output[:200]}


# Execution
prompts = df.apply(build_extraction_prompt, axis=1).tolist()
responses = llm.generate(prompts, sampling_params)

df_ext = pd.DataFrame([robust_json_parse(r.outputs[0].text) for r in responses])

Adding requests:   0%|          | 0/30 [00:00<?, ?it/s]

/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/mistral_common/tokens/tokenizers/sentencepiece.py:133: FutureWarning: `get_control_token` is deprecated. Use `get_special_token` instead.
  warnings.warn("`get_control_token` is deprecated. Use `get_special_token` instead.", FutureWarning)


Processed prompts:   0%|          | 0/30 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [43]:
df_ext.tail()

,relevant,publication_date,status,start_date,end_date,reason,confidence,risk_description
25,True,2022-08-01,Fermeture complète,2022-07-01,Ongoing,Nightly closure due to staff shortage,High,NaN
26,True,2022-05-18,Régulation des entrées,2022-05-18,Ongoing,Lack of staff,High,NaN
27,True,2025-12-01,Fermeture complète,2024-02-01,Ongoing,Night closure and daytime access only on the 15th of each month,High,NaN
28,potential,2025-07-24,NaN,NaN,NaN,NaN,NaN,"The article mentions that the 15 (regulation by the 15) has been generalized in Rennes, a practice that was first implemented in Digne-les-Bains and Manosque. This suggests a potential disruption in the urgences of the CH de Rambouillet in Rennes."
29,True,2023-07-01,Régulation des entrées,2023-07-01,Ongoing,The emergency department is reserved for the most serious cases due to tensions on the urgences and the hospital offer in the context of the cyberattack on the Centre Hospitalier de Versailles.,High,NaN


In [44]:
df_final = pd.concat([df.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)
df_final = pd.concat([df.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)
pd.set_option('display.max_colwidth', None)
df_final.tail()

,finess,hospital_name,nom_etab_long,keywords_nom_etab,source_url,title,content,score,retrieved_at,relevant,publication_date,status,start_date,end_date,reason,confidence,risk_description
25,780000329,CH DE RAMBOUILLET,CENTRE HOSPITALIER DE RAMBOUILLET,RAMBOUILLET,https://www.samu-urgences-de-france.fr/medias/files/sudf_enquete_202207_resultats_VF.pdf,[PDF] RÉSULTATS DE L'ENQUÊTE SUdF SITUATION DES URGENCES ...,"totale de leur UHCD. Les SU ont enregistré durant le mois de juillet 2022 une augmentation d'activité en moyenne de 12,3% soit environ 180.000 passages de plus qu'en 2021 sur la même période. Cette augmentation est de 10% dans les départements où une régulation médicale préalable à l’accès au SU a été mise en place (cf. carte d’augmentation de l’activité des SU). 88 établissements (26%) ont mis en place une restriction d’accès, dont 67 avec une régulation médicale systématique par le Samu-Centre 15 pour autoriser l’accès aux urgences (recommandation n°23). 42 établissements ont été contraints de réaliser une fermeture totale de nuit de leur SU pour un nombre cumulé de 546 nuits en juillet. De jour ce sont 23 établissements qui ont réalisé une fermeture totale pour un nombre cumulé de 208 [...] (sur les 102 du territoire national). Concernant leurs ressources humaines, ils déclarent être en difficulté sur les ressources médicales pour 98% d’entre eux et pour les ressources non médicales pour 95%. 68% de ces établissements ont recours à des solutions d’intérim durant cet été. Les SAMU ont enregistré durant le mois de juillet 2022 une augmentation d'activité en moyenne de 21,5% comparativement à celle de 2021 à la même période. 21 départements ont une augmentation supérieure à 30%, dont 42% ayant mis en place une régulation médicale systématique pour autoriser l’accès aux urgences (cf. carte d’augmentation de l’activité des SAMU). Au total 42 départements (43%) ont mis en œuvre une régulation médicale systématique par le SAMU-Centre 15 avant l’accès aux SU (recommandation n°23). [...] n°27). SUdF - Situation des Urgences Juillet 2022 5 Commentaires : Compte tenu de la suractivité inhabituelle s’ajoutant aux flux estivaux importants dans certaines régions, du déficit majeur de personnels soignants et médicaux, et de l’indisponibilité des lits, très insuffisants pour permettre les hospitalisations nécessaires, les SU sont en très grande fragilité. La mise en œuvre des recommandations de la mission flash est insuffisante et ne permet pas d’assurer une fluidité et un fonctionnement sécuritaire dans le SU. La situation attendue au mois août va encore se dégrader, avec une augmentation des fermetures institutionnelles de lits et une diminution de la disponibilité de l’offre soignante libérale liée aux congés. Sans mesures contraignantes, il faut s’attendre à l’évolution",0.995245,2026-03-14 11:31:23,True,2022-08-01,Fermeture complète,2022-07-01,Ongoing,Nightly closure due to staff shortage,High,NaN
26,780000329,CH DE RAMBOUILLET,CENTRE HOSPITALIER DE RAMBOUILLET,RAMBOUILLET,https://www.franceinfo.fr/economie/greve/greve-aux-urgences/hopital-faute-de-soignants-des-services-durgences-ferment-la-nuit_5145202.html,"Hôpital : faute de soignants, des services d'urgences ferment la nuit","Dès mercredi 18 mai, il sera impossible pour les Bordelais de se rendre dans cet hôpital, entre 20h et 8h du matin, sauf accord préalable du Samu. Le plus important centre hospitalier de la Gironde étouffe. La fréquentation a augmenté de près de 50 % depuis le début de la crise sanitaire​, mais de nombreux médecins urgentistes ont quitté leur poste. Ce système de régulation pourrait réduire le nombre d’entrées d’une trentaine de patients chaque jour​, et pourrait s’inscrire dans la durée. \n\n## Un problème à l’échelle nationale [...] ## Un problème à l’échelle nationale\n\nLa situation est similaire dans le Vaucluse, où les urgences fermeront la nuit, faute de personnels. Les habitants sont inquiets. Selon les syndicats, 66 services d’urgence seraient